# Vision2Slope Tutorial

This tutorial demonstrates how to use Vision2Slope for road slope analysis from street view images.

## 1. Installation (Colab)

Clone the repository and install the package in the Colab runtime.

**Use a GPU runtime** (*Runtime → Change runtime type → T4 GPU*): the Mask2Former segmentation model is very slow on CPU. Vision2Slope needs Python 3.9–3.12.

### 1.1 Clone repository

In [ ]:
# Skip the clone if the repository is already there (e.g. when re-running the notebook)
!test -d /content/Vision2Slope || git clone https://github.com/CubicsYang/Vision2Slope.git /content/Vision2Slope

In [ ]:
%cd /content/Vision2Slope

### 1.2 Install the package

A regular (non-editable) install, so the package can be imported in this session without restarting. Its dependencies come from `requirements.txt`.

If Colab asks you to restart the session after installation, restart it and continue from section 2.

In [ ]:
%pip install /content/Vision2Slope/src/Vision2Slope

## 2. Environment Setup and Imports

First, import the necessary libraries and modules.

In [ ]:
# Move to the repository root so the relative paths below work
%cd /content/Vision2Slope

from pathlib import Path
import torch

# Import Vision2Slope modules
from vision2slope import (
    PipelineConfig,
    VisualizationConfig,
    ProcessingConfig,
    Vision2SlopePipeline
)

print("✓ All libraries imported successfully!")
print(f"Working directory: {Path.cwd()}")
print(f"GPU available: {torch.cuda.is_available()}")

## 3.1. Basic Usage: Processing Panoramic Images

Convert panoramic street view images to perspective views and analyze them. By default this uses the two sample panoramas shipped in `examples/input_pano`; section 3.1.1 can replace it with panoramas downloaded from Mapillary.

In [ ]:
# Bundled sample panoramas; the optional Mapillary download below replaces them
pano_input_dir = "examples/input_pano"

### 3.1.1 (Optional) Download Mapillary Panoramas

Download panoramic street view images from Mapillary for analysis. Leave the placeholder key to skip the download and use the bundled samples.

**Note:** Requires a Mapillary API key. Register at https://www.mapillary.com/

In [ ]:
# Set your Mapillary API key (get it from https://www.mapillary.com/)
mly_api_key = "YOUR_MAPILLARY_API_KEY_HERE"  # Replace with your actual API key

if mly_api_key == "YOUR_MAPILLARY_API_KEY_HERE":
    print("No API key set: skipping the download and using the bundled sample panoramas")
else:
    from zensvi.download import MLYDownloader

    downloader = MLYDownloader(mly_api_key=mly_api_key)

    # Example: Download panoramic images from San Francisco
    downloader.download_svi(
        "sf",
        lat=37.797423890238,
        lon=-122.44403351517,
        buffer=5,
        resolution=2048,
        image_type="pano"  # 'flat', 'pano', or 'all'
    )

    # zensvi puts each download run in a new batch_<n> folder; use the latest one
    batches = sorted(Path("sf/mly_svi").glob("batch_*"), key=lambda p: int(p.name.split("_")[-1]))
    pano_input_dir = str(batches[-1])
    print(f"Using downloaded panoramas in {pano_input_dir}")

### 3.1.2 Process Panoramic Images

Each panorama is rendered into two side views (90° and 270°), and each view is processed separately. Of the two bundled samples, `_7BoWCVskqr-MqAf7r6j8A` is a sloped street where both views succeed. `jrLyG2ebQhnTNhFxTLvE4Q` is a narrow street where both side views face nearby walls, so it gets no slope estimate (`no_lines_detected` and `no_road_detected` in the results CSV). Wider streets with visible road and building edges work best.

In [ ]:
# Configure for panoramic image processing
config_panorama = PipelineConfig(
    input_dir=pano_input_dir,
    output_dir="examples/output_pano",
    processing_config=ProcessingConfig(
        is_panorama=True,           # Enable panorama preprocessing
        panorama_fov=90,            # Field of view (degrees)
        panorama_phi=0.0,           # Vertical angle
        panorama_aspects=(10, 10),  # Aspect ratio
        panorama_show_size=100,     # Scale factor
        log_level="INFO"
    ),
    viz_config=VisualizationConfig(
        save_visualizations=True,
        save_corrected_images=True,
        save_intermediate_results=True
    )
)

print("✓ Panorama pipeline configured")
print(f"  Input:  {config_panorama.input_dir}")
print(f"  Output: {config_panorama.output_dir}")

In [ ]:
# Process panoramic images
pipeline_panorama = Vision2SlopePipeline(config_panorama)
results_panorama = pipeline_panorama.process_batch()

# process_batch returns the successful views kept for slope estimation;
# per-image results (all statuses) are in vision2slope_results_<timestamp>.csv
print("✓ Panorama processing completed!")
if results_panorama.empty:
    print("  No views passed processing and angle filtering; see pipeline.log in the output folder")
else:
    print(f"  Views with slope estimates: {len(results_panorama)}")
    print(f"  Panoramas: {results_panorama['pano_id'].nunique()}")
    display(results_panorama.drop_duplicates('pano_id')[['pano_id', 'road_estimated_slope']])
print(f"Results saved to: {config_panorama.output_dir}")

## 3.2. Basic Usage - Processing Perspective Images

Process regular street view images (non-panoramic format). This uses the 8 sample perspective views in `examples/input`, named `<pano_id>_Direction_<angle>_FOV_...`.

In [ ]:
# Configure for processing perspective images
config_perspective = PipelineConfig(
    input_dir="examples/input",
    output_dir="examples/output"
)

print("✓ Configuration created for perspective images")
print(f"  Input:  {config_perspective.input_dir}")
print(f"  Output: {config_perspective.output_dir}")

In [ ]:
# Run pipeline on perspective images
pipeline_perspective = Vision2SlopePipeline(config_perspective)
results_perspective = pipeline_perspective.process_batch()

# process_batch returns the successful views kept for slope estimation;
# per-image results (all statuses) are in vision2slope_results_<timestamp>.csv
print("✓ Perspective processing completed!")
if results_perspective.empty:
    print("  No views passed processing and angle filtering; see pipeline.log in the output folder")
else:
    print(f"  Views with slope estimates: {len(results_perspective)}")
    print(f"  Panoramas: {results_perspective['pano_id'].nunique()}")
    display(results_perspective.drop_duplicates('pano_id')[['pano_id', 'road_estimated_slope']])
print(f"Results saved to: {config_perspective.output_dir}")